<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Requirements Gathering

The implementation notebook is the executable source of truth. This notebook records the exact inputs, processing stages, outputs, failure handling and validation rules implemented by `camera_calibration.ipynb`.

## Functional Requirements

- load sorted JPEG calibration images from a repository-relative path;
- detect a complete $8 \times 6$ internal-corner chessboard pattern;
- refine detected corners with `cv2.cornerSubPix`;
- skip views whose chessboard cannot be detected;
- create planar calibration coordinates in metres using a $0.03$ m square size;
- normalize image and planar points to zero centroid and mean radius $\sqrt{2}$;
- estimate one homography per valid view using normalized DLT and SVD;
- stack two Zhang constraints per valid homography;
- recover the shared intrinsic matrix $K$;
- recover one pose $(R,t)$ per retained view;
- reproject the calibration points with the recovered pinhole model;
- compute per-point, per-view and global reprojection errors;
- save the six implemented figures under `../outputs/figures/`;
- run final numerical and output-file validation checks.

## Data and Algorithm Requirements

| Requirement | Implemented value |
| --- | --- |
| Input path | `../data/calibration_images` |
| Input extension | `*.jpg` |
| Ordering | `sorted(...)` by filename |
| Internal corners | $8 \times 6$ |
| Points per valid view | 48 |
| Square size | $0.03\,\mathrm{m}$ |
| Minimum valid views | 3 |
| Corner detector | `cv2.findChessboardCorners` |
| Sub-pixel window | $11 \times 11$ |
| Sub-pixel criteria | max 30 iterations, epsilon $0.001$ |
| Point normalization | centroid to origin, mean distance to $\sqrt{2}$ |
| Homography solver | normalized DLT + SVD |
| Intrinsic solver | Zhang constraints + SVD |
| Rotation cleanup | nearest rotation by SVD, determinant $+1$ |
| Error metric | Euclidean pixel reprojection error |
| Distortion model | none |

## Input / Output Contract

### Inputs

`../data/calibration_images/*.jpg`

### In-memory numerical outputs

$H$, $V$, $b$, $K$, per-view $R$ and $t$, projected points, point-wise errors, per-view mean error, per-view RMSE, overall mean error and overall RMSE.

### Saved visual outputs

1. `detected_chessboard_corners.png`
2. `homography_estimation_pipeline.png`
3. `estimated_camera_poses.png`
4. `reprojection_results.png`
5. `mean_reprojection_error_by_view.png`
6. `reprojection_error_distribution.png`

## Implemented Pipeline

```text
Sorted calibration images
        ↓
Path and file validation
        ↓
Chessboard detection + sub-pixel refinement
        ↓
Planar coordinates (Z = 0, square size = 0.03 m)
        ↓
Image/plane point normalization
        ↓
Normalized DLT: Qh = 0
        ↓
Denormalized homography H
        ↓
Zhang constraints: Vb = 0
        ↓
Closed-form intrinsic matrix K
        ↓
Pose recovery R, t
        ↓
3D planar points → camera frame → image projection
        ↓
Point-wise reprojection errors
        ↓
Per-view + global mean / RMSE
        ↓
Six diagnostic figures
        ↓
Final validation checks
```

## Method Selection and Rationale

| Need | Implemented method | Reason |
| --- | --- | --- |
| Corner measurement | OpenCV chessboard detector + sub-pixel refinement | Provides ordered image correspondences at sub-pixel precision |
| Plane-to-image mapping | Normalized DLT | Linear homography estimate with improved conditioning |
| Homogeneous systems | SVD | Returns the least-squares null-space vector |
| Intrinsic calibration | Zhang planar method | Recovers one shared $K$ from multiple planar homographies |
| Pose recovery | $K^{-1}H$ decomposition | Recovers the first two rotation columns and translation |
| Rotation validity | SVD projection | Enforces orthonormality and $\det(R)=+1$ |
| Model evaluation | Reprojection error | Directly measures image-space geometric consistency |

## Requirement-to-Implementation Traceability

| Requirement | Implementation evidence |
| --- | --- |
| Relative input path | `DATA_DIR = Path("../data/calibration_images")` |
| Sorted deterministic images | `sorted(DATA_DIR.glob("*.jpg"))` |
| $8 \times 6$, $0.03$ m target | `INTERNAL_CORNERS_X = 8`, `INTERNAL_CORNERS_Y = 6`, `SQUARE_SIZE_M = 0.03` |
| Minimum 3 views | `MIN_VALID_VIEWS = 3` and runtime check |
| Sub-pixel corners | `cv2.cornerSubPix(..., (11,11), ..., 30, 0.001)` |
| Point normalization | `normalize_trans` |
| Normalized DLT | `Image.find_homography` |
| Zhang constraints | `Image.construct_v` and stacked `V` |
| Intrinsics | closed-form recovery of $\alpha,\beta,\gamma,u_0,v_0$ and $K$ |
| Pose | `Image.find_extrinsic(K)` |
| Reprojection | `camera_points`, `projected_h`, `projected_pixels` |
| Metrics | `errors`, `mean_error`, `rmse`, `overall_mean_error`, `overall_rmse` |
| Figures | six calls to `fig.savefig(...)` |
| Final checks | finite $K$, proper $R$, finite residuals, required files present |

## Failure Modes and Implemented Handling

| Failure | Implemented handling |
| --- | --- |
| Missing image directory | `FileNotFoundError` |
| No JPEG images | `FileNotFoundError` |
| Unreadable image | `FileNotFoundError` inside `Image` |
| Chessboard not detected | `ValueError`, view is skipped |
| Fewer than 3 valid views | `RuntimeError` |
| Coincident points during normalization | `ValueError` |
| Degenerate homography scale | `ValueError` |
| Degenerate intrinsic denominator | `ValueError` |
| Invalid $K$ | final finite/shape validation |
| Non-orthonormal or improper $R$ | final validation error |
| Non-finite residuals | final validation error |
| Missing diagnostic figures | `FileNotFoundError` |

## Scope Boundaries

The implementation intentionally does **not** estimate radial or tangential distortion and does not perform nonlinear refinement after Zhang's closed-form calibration. The documented solution therefore describes the implemented pinhole-model workflow only.

## Implementation Order

The executable notebook follows this order:

1. Environment and Imports
2. Configuration
3. Paths
4. Core Functions / Classes
5. Data Loading
6. Data Validation
7. Pipeline Implementation
8. Execution
9. Results
10. Quantitative Evaluation
11. Visual Evaluation
12. Save Outputs
13. Validation Checks
14. Final Result Summary